In [3]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
load_dotenv()
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so

### Downloading data about visits and registrations and creating json file with visits to registrations conversion(%)

In [4]:
DATE_BEGIN = os.getenv("DATE_BEGIN")
DATE_END = os.getenv("DATE_END")
API_URL = os.getenv("API_URL")

In [ ]:
Visits = requests.get(f"{API_URL}/visits", params={"begin": DATE_BEGIN, "end": DATE_END})
regs = requests.get(f"{API_URL}/registrations", params={"begin": DATE_BEGIN, "end": DATE_END})
Visits = pd.DataFrame(Visits.json())
regs = pd.DataFrame(regs.json())

In [ ]:
visits_cleaned = Visits[Visits["platform"] != "bot"]
visits_sorted = visits_cleaned.sort_values(by="datetime")
visits_sorted = visits_cleaned.drop_duplicates(subset=["visit_id"], keep="last")

In [ ]:
United = pd.merge(visits_sorted, regs, on=["datetime", "platform"], how="outer")
United["datetime"] = United["datetime"].str.slice(0,10)

In [ ]:
conversion = United.groupby(["datetime", "platform"], as_index=False).agg(visits = ("visit_id", "count"), registrations = ("user_id", "count"))
conversion = conversion.rename(columns = {"datetime": "date_group"})
conversion["conversion"] = conversion["registrations"] / conversion["visits"] * 100

In [ ]:
conversion.to_json("./conversion.json")

### Creating dataframe with visits, registrations and ad campaigns

In [ ]:
ads = pd.read_csv("./ads.csv")
ads["date"] = ads["date"].str.slice(0,10)
ads = ads.rename(columns = {"date": "date_group"})
purified_conv = conversion.drop(["platform", "conversion"], axis=1)

In [ ]:
United_ads_conv = pd.merge(purified_conv, ads, on=["date_group"], how="outer")
United_ads_conv = United_ads_conv.groupby("date_group", as_index=False).agg("sum")
United_ads_conv = United_ads_conv[United_ads_conv["date_group"] < DATE_END]
United_ads_conv = United_ads_conv.drop(["utm_source", "utm_medium"], axis=1)
United_ads_conv["utm_campaign"] = United_ads_conv["utm_campaign"].replace(0, "none")

In [ ]:
United_ads_conv.to_json("./ads.json")

### Creating graphes for visit, registrations, conversion and ads

In [ ]:
conversion["conversion"] = conversion["conversion"].round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(40, 10))
sns.barplot(x="date_group", y="visits", data=United_ads_conv).set(title="Total visits")
for container in ax.containers:
    ax.bar_label(container)
ax.xaxis.set_tick_params(rotation=60)
plt.savefig('./charts/Total_Visits.png', bbox_inches="tight")

In [ ]:
f, ax = plt.subplots(figsize= (40,10))
ax.xaxis.set_tick_params(rotation=60)
plt.tight_layout()
p = (
    so.Plot(conversion, x="date_group", y="visits", color="platform")
    .add(so.Bar(), so.Agg(func="sum"), so.Stack())
    .label(title="Visits by Platform (Stacked)")
    .theme({"grid.color": "black", "axes.facecolor": "white"})
    .on(ax)
)
p.save('./charts/Visits_by_platform.png', bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(40, 10))
sns.barplot(x="date_group", y="registrations", data=United_ads_conv).set(title="Total Registrations")
for container in ax.containers:
    ax.bar_label(container)
ax.xaxis.set_tick_params(rotation=60)
plt.savefig('./charts/Total_Registrations.png', bbox_inches="tight")


In [ ]:
f, ax = plt.subplots(figsize= (40,10))
ax.xaxis.set_tick_params(rotation=60)
plt.tight_layout()
p = (
    so.Plot(conversion, x="date_group", y="registrations", color="platform")
    .add(so.Bar(), so.Agg(func="sum"), so.Stack())
    .label(title="Registrations by Platform (Stacked)")
    .theme({"grid.color": "black", "axes.facecolor": "white"})
    .on(ax)
)
p.save('./charts/Registrations_by_platform.png', bbox_inches="tight")

In [ ]:
all_platforms = conversion.drop("platform", axis=1).groupby("date_group", as_index=False).agg("sum")
all_platforms["conversion"] = (all_platforms["registrations"]/all_platforms["visits"] * 100).round(0)
all_platforms["conversion"] = all_platforms["conversion"].astype(int)
fig, ax = plt.subplots(figsize=(40, 10))
sns.lineplot(data=all_platforms, x="date_group", y="conversion", marker="o").set(title="Overall Conversion", xlabel="Date", ylabel="Conversion (%)")
ax.xaxis.set_tick_params(rotation=60)
plt.grid()
for x, y in zip(all_platforms["date_group"], all_platforms["conversion"]):
    ax.text(x, y + 0.5, f'{y} %', ha='center', va='bottom', fontsize=8)
plt.savefig('./charts/Overall_Conversion.png', bbox_inches="tight")

In [ ]:
conversion_android = conversion[conversion["platform"] == "android"]
conversion_ios = conversion[conversion["platform"] == "ios"]
conversion_web = conversion[conversion["platform"] == "web"]
conversion_android["conversion"] = conversion_android["conversion"].astype(int)
conversion_ios["conversion"] = conversion_ios["conversion"].astype(int)
conversion_web["conversion"] = conversion_web["conversion"].astype(int)

fig, axes = plt.subplots(3, 1, constrained_layout=True)
fig.set_size_inches(30, 15)

axes[0].plot("date_group", "conversion", data=conversion_android)
axes[0].set_title("Conversion android")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Conversion (%)")
axes[0].xaxis.set_tick_params(rotation=60)
axes[0].grid()
for x, y in zip(conversion_android["date_group"], conversion_android["conversion"]):
    axes[0].text(x, y + 0.5, f'{y} %', ha='center', va='bottom', fontsize=6)

axes[1].plot("date_group", "conversion", data=conversion_ios)
axes[1].set_title("Conversion ios")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Conversion (%)")
axes[1].xaxis.set_tick_params(rotation=60)
axes[1].grid()
for x, y in zip(conversion_ios["date_group"], conversion_ios["conversion"]):
    axes[1].text(x, y + 0.5, f'{y} %', ha='center', va='bottom', fontsize=6)

axes[2].plot("date_group", "conversion", data=conversion_web)
axes[2].set_title("Conversion web")
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Conversion (%)")
axes[2].xaxis.set_tick_params(rotation=60)
axes[2].grid()
for x, y in zip(conversion_web["date_group"], conversion_web["conversion"]):
    axes[2].text(x, y + 0.5, f'{y} %', ha='center', va='bottom', fontsize=6)

plt.savefig('./charts/Conversion_by_platform.png', bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(40, 10))
sns.lineplot(data=United_ads_conv, x="date_group", y="cost", marker="o").set(title="Aggregated Ad Campaign Costs", xlabel="Date", ylabel="Cost (RUB)")
ax.xaxis.set_tick_params(rotation=60)
plt.grid()
for x, y in zip(United_ads_conv["date_group"], United_ads_conv["cost"]):
    if y != 0:
        ax.text(x, y + 2, f'{y} RUB', ha='center', va='bottom', fontsize=6)
plt.savefig('./charts/Aggregated_Ad_Campaign_Costs.png', bbox_inches="tight")

In [ ]:
without_none = United_ads_conv[United_ads_conv["utm_campaign"] != "none"]

fig, axes = plt.subplots(2, 1, constrained_layout=True)
fig.set_size_inches(35, 15)

sns.lineplot(data=United_ads_conv, x="date_group", y="visits", marker="o", color='black', ax=axes[0]).set(title="Visits during marketing active days", xlabel="Date", ylabel="Unique Visits")
axes[0].xaxis.set_tick_params(rotation=60)
axes[0].grid()
axes[0].axhline(United_ads_conv["visits"].mean(), color='black', linestyle='--', label="Average Number of Visits")
axes[0].legend()
sns.barplot(data=United_ads_conv,y=1400, x="date_group", hue=without_none["utm_campaign"], alpha=0.8, width=1, dodge=False, ax=axes[0])
axes[0].set_ylim(0, 1400)

sns.lineplot(data=United_ads_conv, x="date_group", y="registrations", marker="o", color='black', ax=axes[1]).set(title="Registrations during marketing active days", xlabel="Date", ylabel="Unique Users")
axes[1].xaxis.set_tick_params(rotation=60)
axes[1].grid()
axes[1].axhline(United_ads_conv["registrations"].mean(), color='black', linestyle='--', label="Average Number of Registration")
axes[1].legend()
sns.barplot(data=United_ads_conv,y=250, x="date_group", hue=without_none["utm_campaign"], alpha=0.8, width=1, dodge=False, ax=axes[1])
axes[1].set_ylim(0, 250)

plt.savefig('./charts/Visits_and_Registrations_during_marketing_active_days.png', bbox_inches="tight")